# 02 - Regularization-based strategies: EWC, Synaptic Intelligence, LwF

These strategies keep a single growing model and add a penalty/distillation
term so that learning new experiences doesn't destroy what mattered for
old ones:

- **EWC** (Elastic Weight Consolidation): penalizes moving parameters that
  were important (high Fisher information) for previous tasks.
- **Synaptic Intelligence (SI)**: tracks per-parameter importance *online*
  during training instead of a separate Fisher-information pass.
- **LwF** (Learning without Forgetting): distills the previous model's
  predictions into the new model as a soft-label regularizer.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import warnings; warnings.filterwarnings("ignore")

import torch
import pandas as pd
from avalanche.models import SimpleMLP
from avalanche.training import EWC, SynapticIntelligence, LwF
from avalanche.training.plugins import EvaluationPlugin
from avalanche.evaluation.metrics import accuracy_metrics, forgetting_metrics, loss_metrics

from bench_utils import make_synthetic_benchmark
from run_utils import run_strategy

BENCHMARK_CONFIG = dict(
    n_classes=10, n_experiences=5, feature_dim=64,
    n_per_class=250, class_sep=1.6, noise=1.0, seed=0,
)
benchmark = make_synthetic_benchmark(**BENCHMARK_CONFIG)

def new_model():
    return SimpleMLP(num_classes=benchmark.n_classes, input_size=benchmark.feature_dim,
                      hidden_size=64, hidden_layers=1, drop_rate=0.0)

def new_evaluator():
    return EvaluationPlugin(
        accuracy_metrics(experience=True, stream=True),
        forgetting_metrics(experience=True, stream=True),
        loss_metrics(stream=True),
        loggers=[],
    )

all_rows = []


In [2]:
# --- EWC ---
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = EWC(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
               ewc_lambda=0.4, train_mb_size=32, train_epochs=3, eval_mb_size=128,
               evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "EWC", "regularization")
all_rows += rows
print(final)


{'strategy': 'EWC', 'category': 'regularization', 'after_experience': 4, 'stream_acc': 0.266, 'stream_forgetting': 0.9095744680851063, 'train_seconds': 0.591606616973877}


In [3]:
# --- Synaptic Intelligence ---
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = SynapticIntelligence(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
                                 si_lambda=0.4, train_mb_size=32, train_epochs=3, eval_mb_size=128,
                                 evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "SynapticIntelligence", "regularization")
all_rows += rows
print(final)


{'strategy': 'SynapticIntelligence', 'category': 'regularization', 'after_experience': 4, 'stream_acc': 0.344, 'stream_forgetting': 0.8058510638297872, 'train_seconds': 0.6716690063476562}


In [4]:
# --- LwF ---
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = LwF(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
               alpha=1.0, temperature=2.0, train_mb_size=32, train_epochs=3, eval_mb_size=128,
               evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "LwF", "regularization")
all_rows += rows
print(final)


{'strategy': 'LwF', 'category': 'regularization', 'after_experience': 4, 'stream_acc': 0.208, 'stream_forgetting': 0.9867021276595744, 'train_seconds': 0.592557430267334}


In [5]:
df = pd.DataFrame(all_rows)
df.to_csv("../results/02_regularization.csv", index=False)
df


,strategy,category,after_experience,stream_acc,stream_forgetting
0,EWC,regularization,0,0.218,0.000000
1,EWC,regularization,1,0.300,0.486239
2,EWC,regularization,2,0.390,0.515957
3,EWC,regularization,3,0.212,0.961538
4,EWC,regularization,4,0.266,0.909574
5,SynapticIntelligence,regularization,0,0.218,0.000000
6,SynapticIntelligence,regularization,1,0.302,0.477064
7,SynapticIntelligence,regularization,2,0.390,0.515957
8,SynapticIntelligence,regularization,3,0.310,0.804487
9,SynapticIntelligence,regularization,4,0.344,0.805851
